In [ ]:
import espressomd
import espressomd.accumulators
import espressomd.observables
import numpy as np

In [ ]:
# -------------------------------------------------------------------------
# 1. Constants
# -------------------------------------------------------------------------

kT = 1.0
mass = 1.0          # Mass of the tracer particle

gamma_instantaneous = 0.5            # Solvent viscosity (Instantaneous friction)
gamma_memory = 6.5            # Polymer viscosity (Memory friction)
relaxation_time = 10.0           # Relaxation time

STEPS = 3 *1000 * 333

time_step = 0.003
numpy_sim_dt = 0.01

In [ ]:
# -------------------------------------------------------------------------
# 2. ESPResSo System Setup
# -------------------------------------------------------------------------
system = espressomd.System(box_l=[50, 50, 50])
system.time_step = 0.003
system.cell_system.skin = 0.4

system.thermostat.set_jeffreys_langevin(kT=kT,
                                        gamma=gamma_instantaneous,
                                        gamma_retarded=gamma_memory,
                                        relax_time=relaxation_time,
                                        seed=42)

In [ ]:
# -------------------------------------------------------------------------
# 3. Particle Creation
# -------------------------------------------------------------------------
particle = system.part.add(
    pos=[25, 25, 25],
    mass=mass,
)

particle.mass, particle.v, particle.retarded_force

In [ ]:
# -------------------------------------------------------------------------
# 4. Correlators
# -------------------------------------------------------------------------

def msd_correlator(pids, tau_max):
    pos = espressomd.observables.ParticlePositions(ids=pids)
    pos_corr = espressomd.accumulators.Correlator(
        obs1=pos, tau_lin=16, tau_max=tau_max, delta_N=1,
        corr_operation="square_distance_componentwise", compress1="discard1")
    return pos_corr

def vel_correlator(pids, tau_max):
    vel = espressomd.observables.ParticleVelocities(ids=pids)
    vel_corr = espressomd.accumulators.Correlator(
        obs1=vel, tau_lin=16, tau_max=tau_max, delta_N=1,
        corr_operation="scalar_product", compress1="discard1")
    return vel_corr

In [ ]:
# -------------------------------------------------------------------------
# 5. Simulation
# -------------------------------------------------------------------------
print("Equilibrating the system.")
system.integrator.run(10000)
print("Equilibration finished.")

correlator_msd = msd_correlator([particle.id], STEPS)
correlator_vel = vel_correlator([particle.id], STEPS)
system.auto_update_accumulators.add(correlator_msd)
system.auto_update_accumulators.add(correlator_vel)


print("Simulating Jeffreys Fluid Brownian Motion...")
system.integrator.run(STEPS)

correlator_msd.finalize()
correlator_vel.finalize()
tau_results = correlator_msd.lag_times()
msd_results = np.sum(correlator_msd.result().reshape([-1, 3]), axis=1)
vacf_results = np.sum(correlator_vel.result().reshape([-1, 1]), axis=1)
# In our setup, both correlators should produce values for the same lag times,
# we therefore do not have to save the lag times twice ...
assert np.array_equal(tau_results, correlator_vel.lag_times())
system.auto_update_accumulators.clear()
system.thermostat.turn_off()

In [ ]:
tau_results.shape

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

start, end = 0, 120

plt.rcParams.update({'font.size': 18})

plt.figure(figsize=(10, 6))
plt.xlabel(r'$\tau$ [$\Delta t$]')
plt.ylabel(r'MSD [$\sigma^2$]')


plt.loglog(tau_results[start:end], 6*kT/(gamma_instantaneous + gamma_memory)*tau_results[start:end], linestyle='dashed',
            color="red", label=fr'total diffusion ($\gamma={gamma_instantaneous + gamma_memory:.1f}$)')

plt.loglog(tau_results[start:end], 3*kT/(mass)*(tau_results[start:end] ** 2), linestyle='dashdot',
            color="green", label=fr'ballistic')

plt.loglog(tau_results[start:end], msd_results[start:end], label=fr'simulation')

plt.title("MSD by implemented simulation")
plt.legend(ncol=2, columnspacing=0.5, handlelength=1.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.xlabel(r"$\tau$ [$\Delta t$]")
plt.ylabel(r"$\langle {\bf v}(t_0) {\bf v}(t_0 + \tau) \rangle$")

plt.semilogx(tau_results[start:end], vacf_results[start:end])
plt.title("velocity autocovariance by implemented")
plt.show()

## Compare with numpy-based

In [ ]:
def simulate_jeffreys_brownian_motion(
        kT = kT,
        m = mass,
        gamma_s=gamma_instantaneous,
        gamma_p=gamma_memory, # aka gamma_polimer
        tau=relaxation_time,
        N_particles=500,
        T_max=150.0,
        dt=0.01,
        ndim=1,
    ):

    N_steps = int(T_max / dt)
    time = np.linspace(0, T_max, N_steps)
    
    x = np.zeros((N_steps, N_particles, ndim))
    v = np.zeros((N_steps, N_particles, ndim))
    z = np.zeros((N_steps, N_particles, ndim)) # The auxiliary elastic force
    
    # Initial conditions (start from rest at origin)
    x[0, :] = 0.0
    v[0, :] = 0.0 # Or sample from Maxwell-Boltzmann: np.random.normal(0, np.sqrt(kB_T/m), N_particles)
    z[0, :] = 0.0 
    
    # Pre-generate Random Noise (Gaussian Normal Distribution)
    # Scale by sqrt(dt) for Euler-Maruyama
    # We need TWO independent noise sources
    noise_s = np.random.normal(0, np.sqrt(dt), (N_steps, N_particles, ndim))
    noise_p = np.random.normal(0, np.sqrt(dt), (N_steps, N_particles, ndim))
    
    # --- 3. The Euler-Maruyama Integration Loop ---
    # We define constants here to speed up the loop
    sqrt_2_kbT_gs = np.sqrt(2 * kT * gamma_s)
    sqrt_2_kbT_gp = np.sqrt(2 * kT * gamma_p)
    
    print("Starting simulation...")
    
    for i in range(N_steps - 1):
        # Current state
        v_curr = v[i, :]
        z_curr = z[i, :]
        
        # 1. Update Position
        # dx = v * dt
        x[i+1, :] = x[i, :] + v_curr * dt
        
        # 2. Update Velocity (Equation 2)
        # m*dv = (-gamma_s * v + z) * dt + noise
        deterministic_force_v = -gamma_s * v_curr + z_curr
        stochastic_force_v = sqrt_2_kbT_gs * noise_s[i, :]
        
        dv = (deterministic_force_v * dt + stochastic_force_v) / m
        v[i+1, :] = v_curr + dv
        
        # 3. Update Internal Stress (Equation 3 - The Memory)
        # dz = -(1/tau)*(z + gamma_p*v) * dt + (1/tau)*noise
        deterministic_force_z = -(1.0/tau) * (z_curr + gamma_p * v_curr)
        stochastic_force_z = (1.0/tau) * sqrt_2_kbT_gp * noise_p[i, :] # probably not a force, but an impulse
        
        dz = deterministic_force_z * dt + stochastic_force_z
        z[i+1, :] = z_curr + dz

    return time, x



# --- 4. Run and Validate ---
time, trajectories = simulate_jeffreys_brownian_motion(dt=numpy_sim_dt, ndim=3)

skip_n_steps = 1000

time = time[skip_n_steps:] - time[[skip_n_steps]]
trajectories = trajectories[skip_n_steps:] - trajectories[[skip_n_steps]]

# Calculate Mean Squared Displacement (MSD)
# MSD = <(x(t) - x(0))^2>
msd = np.mean(np.sum(trajectories**2, axis=-1), axis=1)


# Plotting
plt.figure(figsize=(10, 6))

# Plot a few individual trajectories
# plt.subplot(2, 1, 1)
plt.plot(time, trajectories[:, :5, 0], alpha=0.7)
plt.title("Individual Particle Trajectories (1d slice)")
plt.ylabel("Position")
plt.xlabel("Time")


plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.xlabel(r'$\tau$ [$\Delta t$]')
plt.ylabel(r'MSD [$\sigma^2$]')

# Theoretical comparison for simple diffusion (Einstein relation)
# At very long times, Jeffreys fluid behaves like a viscous fluid with total viscosity = gamma_instantaneous + gamma_memory
plt.loglog(time, 6*kT/(gamma_instantaneous + gamma_memory) * time, linestyle='dashed',
            color="red", label=fr'total diffusion ($\gamma={gamma_instantaneous + gamma_memory:.1f}$)')

# At very short times, particle experience few collisions and inertia plays a big role
plt.loglog(time, 3*kT/(mass)*(time ** 2), linestyle='dashdot',
            color="green", label=fr'ballistic')

plt.loglog(time, msd, label=fr'simulated')

plt.title('MSD by numpy simulation')
plt.legend(ncol=2, columnspacing=0.5, handlelength=1.3)
plt.show()

In [ ]:
v_log = (trajectories[1:] - trajectories[:-1]) / numpy_sim_dt
v_time_cov = (v_log * v_log[0]).sum(axis=2).mean(axis=1)

plt.figure(figsize=(10, 6))
plt.xlabel(r"$\tau$ [$\Delta t$]")
plt.ylabel(r"$\langle {\bf v}(t_0) {\bf v}(t_0 + \tau) \rangle$")

plt.semilogx(time[:-1], v_time_cov)
plt.title("velocity autocovariance by numpy-based")
plt.show()

## Compare implemented with numpy-based

In [ ]:
plt.figure(figsize=(10, 6))
plt.xlabel(r'$\tau$ [$\Delta t$]')
plt.ylabel(r'MSD [$\sigma^2$]')

plt.loglog(tau_results[start:end], msd_results[start:end], label=fr'implemented')
plt.loglog(time, msd, label='numpy-based')

plt.title("MSD by both simulations")
plt.legend(ncol=2, columnspacing=0.5, handlelength=1.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.xlabel(r"$\tau$ [$\Delta t$]")
plt.ylabel(r"$\langle {\bf v}(t_0) {\bf v}(t_0 + \tau) \rangle$")

plt.semilogx(tau_results[start:end], vacf_results[start:end], label='implemented')
plt.semilogx(time[:-1], v_time_cov, label=fr'numpy-based')
plt.title('velocity autocovariance')
plt.legend()
plt.show()